# 00 · Datos y selección de marcas

**Objetivo:** Mostrar a revenue management y category management cómo se comporta en el anaquel la arquitectura de precios por tamaño de marcas líderes, y dónde se rompe.

**Pregunta:** En la misma tienda, ¿el precio por litro, kilo o pieza baja al subir el tamaño en marcas líderes?

Este notebook responde: ¿qué marcas tienen varios tamaños del mismo producto y la misma variante en la misma
tienda, y cuáles se analizan? Cierra con la tabla de decisiones metodológicas (plan aprobado en
`_specs/plan_metodologico_6_12.md`).

**Escalón** = un tamaño contra el siguiente más grande que tiene la tienda, comparando el precio por litro, kilo o pieza.
**Escalón roto** = el tamaño grande cuesta más por unidad que el chico.

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import entorno
entorno.setup()
import profeco, tablas, pares, estadistica as E, charts as C
import escalera as S
PROC, META, CHARTS, SOCIAL = entorno.PROC, entorno.META, entorno.CHARTS, entorno.SOCIAL
CFG = entorno.config()
reg = entorno.registrar
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)
pts, st, cat = S.load()
REPS = 500
raw = entorno.load_raw()
manifest = entorno.write_manifest(raw)
print(f"Precios de tienda en supermercados: {len(pts):,} · tiendas: {pts.establecimiento_id.nunique()}")
sel = S.selection(pts, st, cat)
display(sel[["escalera", "categoria", "unidad", "tamanos", "tiendas_3_tamanos", "seleccionada", "presentaciones"]])

Precios de tienda en supermercados: 306,044 · tiendas: 319


,escalera,categoria,unidad,tamanos,tiendas_3_tamanos,seleccionada,presentaciones
0,Coca-Cola,Refresco,litro,4,206,True,Lata 355 Ml.; Botella 600 Ml.; Botella 1 Lt. No Retornable; Botella 2.75 Lt. No Retorn...
1,Huevo San Juan,Huevo,pieza,3,176,True,Paquete C/12 Blanco; Paquete C/18 Blanco; Paquete C/30 Blanco
2,Huevo El Calvario,Huevo,pieza,3,121,True,Paquete C/12 Blanco; Paquete C/18 Blanco; Paquete C/30 Blanco
3,Zucaritas,Cereal,kilo,4,141,True,Caja 30 Gr.; Caja 380 Gr.; Caja 600 Gr.; Caja 620 Gr.
4,Chiles La Costeña en rajas,Enlatados,kilo,5,192,True,Lata 105 Gr. Jalapeños Rajas; Lata 220 Gr. Jalapeños Rajas; Lata 240 Gr. Jalapeños Raj...
5,Agua Ciel,Agua,litro,3,74,False,Botella 600 Ml.; Botella 1 Lt.; Botella 1.5 Lt.
6,Galletas Saladitas,Galletas,kilo,3,2,False,Paquete 140 Gr. Saladitas; Paquete 186 Gr. Saladitas; Caja 504 Gr. Saladitas
7,Caldo Knorr,Condimentos,kilo,3,2,False,Frasco 450 Gr. Caldo; Bolsa 750 Gr. Caldo; Bolsa 950 Gr. Caldo
8,Pan Bimbo blanco,Pan,kilo,2,0,False,Paquete Chico 350 Gr. Rebanado Blanco; Paquete Grande 620 Gr. Rebanado Blanco
9,Yoghurt Lala fresa (vaso y bote),Lácteos,kilo,2,0,False,Vaso 120 Gr. Fresa; Bote 900 Gr. Fresa


## 1. Selección

**Decisión:** una escalera entra si tiene **3 tamaños o más de la misma variante** (mismo sabor, color o tipo, según el
texto de la presentación) y **100 tiendas o más con al menos 3 de esos tamaños**, en categorías distintas.

- **Entran:** Coca-Cola regular (lata 355 ml, 600 ml, 1 L y 2.75 L), huevo blanco San Juan (12, 18 y 30 piezas),
  Zucaritas (30, 380, 600 y 620 g) y chiles jalapeños en rajas La Costeña (105 a 380 g).
- **Huevo blanco El Calvario** entra como segunda marca de la misma categoría: sirve para comprobar que lo que pase con
  el huevo no es cosa de una sola marca.
- **No entran, aunque el plan las mencionaba como ejemplo:**
  - pan de caja Bimbo: solo 2 tamaños de la misma variante; el de 210 g es pan tostado, otro producto;
  - yoghurt Lala (y Alpura y Yoplait): el tamaño intermedio es bebible, otro producto; vaso y bote son solo 2 tamaños;
  - café Nescafé: cada variante tiene solo 2 tamaños.
- **No alcanzan 100 tiendas con 3 tamaños:** agua Ciel (74), galletas Saladitas y caldo Knorr (2 cada una; casi
  ninguna tienda tiene los tres tamaños a la vez).

La variante se define por el texto de la presentación y se revisó marca por marca (tabla de arriba). Las
presentaciones ambiguas («800 ó 880 g») quedan fuera (`sku_catalogo.presentacion_ambigua`).

**Propietaria:** solo Coca-Cola tiene empresa propietaria verificada en `sku_catalogo`; para las demás marcas no se
menciona la empresa.

In [2]:
SEL = sel[sel.seleccionada].escalera.tolist()
print("Seleccionadas:", SEL)
p = pd.concat([S.steps(pts, st, cat, k) for k in SEL], ignore_index=True)
print(f"Escalones: {len(p):,} · tiendas: {p.establecimiento_id.nunique()} · municipios: {p.municipio_id.nunique()}")
display(p.groupby(["escalera", "escalon"]).agg(escalones=("cambio_pct", "size"), contiguo=("contiguo", "first")))

Seleccionadas: ['Coca-Cola', 'Huevo San Juan', 'Huevo El Calvario', 'Zucaritas', 'Chiles La Costeña en rajas']


Escalones: 2,138 · tiendas: 300 · municipios: 63


escalones  contiguo
escalera                   escalon                               
Chiles La Costeña en rajas 105 g → 220 g             76      True
                           105 g → 240 g            134     False
                           105 g → 340 g              1     False
                           105 g → 380 g              8     False
                           220 g → 240 g              1      True
                           220 g → 340 g              1     False
                           220 g → 380 g            113     False
                           240 g → 340 g            120      True
                           240 g → 380 g             12     False
                           340 g → 380 g              3      True
Coca-Cola                  1 L → 2.75 L             150      True
                           355 ml → 1 L               3     False
                           355 ml → 2.75 L            1     False
                           355 ml → 600 ml          247      True
                           600 ml → 1 L             199      True
                           600 ml → 2.75 L           11     False
Huevo El Calvario          12 pzas → 18 pzas        126      True
                           12 pzas → 30 pzas          8     False
                           18 pzas → 30 pzas        128      True
Huevo San Juan             12 pzas → 18 pzas        199      True
                           12 pzas → 30 pzas          9     False
                           18 pzas → 30 pzas        191      True
Zucaritas                  30 g → 380 g             143      True
                           30 g → 600 g               8     False
                           30 g → 620 g               4     False
                           380 g → 600 g            137      True
                           380 g → 620 g            102     False
                           600 g → 620 g              3      True

## 2. Calidad

- **Precio de tienda** = mediana de la quincena (`precio_tienda_sku`).
- **Extremos:** se marca cada precio con `estadistica.flag_prices`. Un escalón usa dos precios; la sensibilidad
  quita los escalones donde alguno de los dos es extremo.
- **Escalones no contiguos:** si la tienda no tiene el tamaño intermedio, el motor compara con el siguiente que sí
  tiene (p. ej., 12 contra 30 piezas). La sensibilidad usa solo escalones contiguos.

In [3]:
ext = pts.set_index(["establecimiento_id", "sku_id"]).extremo
y = pd.concat([S.skus(cat, k).assign(escalera=k) for k in SEL])
idx = y.set_index(["escalera", "cantidad_base"]).sku_id
p["extremo"] = [bool(ext.get((e, idx.get((k, a))), False) or ext.get((e, idx.get((k, b))), False))
                for e, k, a, b in zip(p.establecimiento_id, p.escalera, p.tam_a, p.tam_b)]
print(f"Escalones con algún precio extremo: {p.extremo.mean() * 100:.1f}% · no contiguos: {(~p.contiguo).mean() * 100:.1f}%")
p.to_csv(PROC / "escalones.csv", index=False, encoding="utf-8-sig")
sel.to_csv(META / "seleccion_marcas.csv", index=False, encoding="utf-8-sig")
params = {"semilla": S.SEMILLA, "bootstrap_replicas": REPS, "canal": S.CANAL, "min_tiendas_3_tamanos": S.MIN_TIENDAS3,
          "plano_pct": S.PLANO, "candidatas": {k: list(v) for k, v in S.CANDIDATAS.items()}, "seleccionadas": SEL}
_ = (META / "parametros.json").write_text(json.dumps(params, ensure_ascii=False, indent=2), encoding="utf-8")
F = "00_datos_y_seleccion"
reg("escalones", len(p), fuente=F); reg("tiendas", p.establecimiento_id.nunique(), fuente=F)
reg("municipios", p.municipio_id.nunique(), fuente=F); reg("marcas", len(SEL), fuente=F)
for k in SEL:
    reg("t3_" + S.KEY[k], sel.set_index("escalera").loc[k, "tiendas_3_tamanos"], fuente=F)
reg("ciel_t3", sel.set_index("escalera").loc["Agua Ciel", "tiendas_3_tamanos"], fuente=F)

Escalones con algún precio extremo: 0.2% · no contiguos: 19.4%


{'escalones': {'cifra': 2138.0,
  'intervalo': None,
  'n': None,
  'fuente': '00_datos_y_seleccion',
  'nota': ''},
 'tiendas': {'cifra': 300.0,
  'intervalo': None,
  'n': None,
  'fuente': '00_datos_y_seleccion',
  'nota': ''},
 'municipios': {'cifra': 63.0,
  'intervalo': None,
  'n': None,
  'fuente': '00_datos_y_seleccion',
  'nota': ''},
 'marcas': {'cifra': 5.0,
  'intervalo': None,
  'n': None,
  'fuente': '00_datos_y_seleccion',
  'nota': ''},
 't3_coca': {'cifra': 206.0,
  'intervalo': None,
  'n': None,
  'fuente': '00_datos_y_seleccion',
  'nota': ''},
 't3_sj': {'cifra': 176.0,
  'intervalo': None,
  'n': None,
  'fuente': '00_datos_y_seleccion',
  'nota': ''},
 't3_ec': {'cifra': 121.0,
  'intervalo': None,
  'n': None,
  'fuente': '00_datos_y_seleccion',
  'nota': ''},
 't3_zuc': {'cifra': 141.0,
  'intervalo': None,
  'n': None,
  'fuente': '00_datos_y_seleccion',
  'nota': ''},
 't3_raj': {'cifra': 192.0,
  'intervalo': None,
  'n': None,
  'fuente': '00_datos_y_selec

## 3. Decisiones metodológicas

In [4]:
display(Markdown(f'''
| # | Decisión | Propuesta | Por qué | Alternativas descartadas |
|---|---|---|---|---|
| 1 | Selección | 3 tamaños o más de la misma variante en 100 tiendas o más, categorías distintas: {len(SEL)} escaleras | escalones comparables y con cobertura | marcas «conocidas» sin revisar variante (Bimbo y yoghurt mezclan productos) |
| 2 | Comparación | misma tienda, mismo producto y variante (`pares.tamanos`); precio por litro, kilo o pieza | lo que ve el cliente frente al anaquel | promedios nacionales por tamaño (mezclan tiendas) |
| 3 | Escalón | cada tamaño contra el siguiente más grande que tiene la tienda | es la decisión real del cliente | todos contra todos (cuenta doble) |
| 4 | Ruptura | el grande cuesta más por unidad; además «plano» = descuento menor a 5% | una escalera rota cobra más por comprar más | solo la pendiente de una regresión precio–tamaño (queda descriptiva) |
| 5 | Incertidumbre | IC90 del % roto por bootstrap de dos vías (escalones y municipios), 500 réplicas, semilla 2026 | las tiendas de una ciudad no son independientes | IC por escalón |
| 6 | Desglose | escalera, escalón, cadena y formato con 20 escalones o más | ¿dónde se rompe más? | — |
| 7 | Calidad | extremos y escalones no contiguos marcados; % roto repetido sin ellos | no borrar en silencio | borrar extremos |
'''))


| # | Decisión | Propuesta | Por qué | Alternativas descartadas |
|---|---|---|---|---|
| 1 | Selección | 3 tamaños o más de la misma variante en 100 tiendas o más, categorías distintas: 5 escaleras | escalones comparables y con cobertura | marcas «conocidas» sin revisar variante (Bimbo y yoghurt mezclan productos) |
| 2 | Comparación | misma tienda, mismo producto y variante (`pares.tamanos`); precio por litro, kilo o pieza | lo que ve el cliente frente al anaquel | promedios nacionales por tamaño (mezclan tiendas) |
| 3 | Escalón | cada tamaño contra el siguiente más grande que tiene la tienda | es la decisión real del cliente | todos contra todos (cuenta doble) |
| 4 | Ruptura | el grande cuesta más por unidad; además «plano» = descuento menor a 5% | una escalera rota cobra más por comprar más | solo la pendiente de una regresión precio–tamaño (queda descriptiva) |
| 5 | Incertidumbre | IC90 del % roto por bootstrap de dos vías (escalones y municipios), 500 réplicas, semilla 2026 | las tiendas de una ciudad no son independientes | IC por escalón |
| 6 | Desglose | escalera, escalón, cadena y formato con 20 escalones o más | ¿dónde se rompe más? | — |
| 7 | Calidad | extremos y escalones no contiguos marcados; % roto repetido sin ellos | no borrar en silencio | borrar extremos |


In [5]:
display(Markdown(
    f"**Conclusión.** Cumplen la regla {len(SEL)} escaleras: " + ", ".join(SEL) + f". Suman **{len(p):,} escalones** en "
    f"{p.establecimiento_id.nunique()} supermercados de {p.municipio_id.nunique()} municipios. Bimbo, yoghurt y café no tienen 3 tamaños de "
    "la misma variante."))

**Conclusión.** Cumplen la regla 5 escaleras: Coca-Cola, Huevo San Juan, Huevo El Calvario, Zucaritas, Chiles La Costeña en rajas. Suman **2,138 escalones** en 300 supermercados de 63 municipios. Bimbo, yoghurt y café no tienen 3 tamaños de la misma variante.